# Supplementary Data Table 2

Environment: env1

In [3]:
import pandas as pd
import os
import shutil
import numpy as np
import pickle
from scipy.stats import fisher_exact, mannwhitneyu

### CU

In [4]:
cu_ids = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/original_manuscript_cohort.pkl')
cu_pec = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_pec_20251017_55.pkl')

ght_cases = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/ght_cases_20250926.pkl')
ght_cases = [x.lower() for x in ght_cases]

cu_md = '/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/Updated_Cases_09_26_2025(All_subjects_09_2025).csv'
cu_md = pd.read_csv(cu_md)
cu_md['Study ID'] = cu_md['Study ID'].str.lower()
cu_md = cu_md[cu_md['Study ID'].isin(cu_ids)]

clinical_md = pd.read_csv('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv')

clinical_md = clinical_md[clinical_md['id'].isin(cu_ids)]

In [19]:
print('pec')
print(len(cu_pec))
print('ghtn')
print(len(ght_cases))
print('gdm')
print(len(cu_md[cu_md['Diagnoses, Maternal'].str.lower().str.contains('diabetes', na=False)]))
print('stillbirth')
print(len(cu_md[cu_md['Baby A Outcome of Delivery']=='Stillborn']))
print('chtn')
print(len(cu_md[cu_md['Cardiac Disease'].str.lower().str.contains('hypertensive', na=False)]))
print()
print('past pec')
print(len(cu_md[cu_md['Past Pregnancies Diagnoses, Maternal, Hypertension']=='Preeclampsia']))
print('ivf')
print(clinical_md['Maternal Diag IVF'].sum())
print('age')
print(clinical_md['Age at enrollment'].mean(),clinical_md['Age at enrollment'].std())
print('first preg')
print(len(cu_md[cu_md['Living Children']==0]))
print('obesity')
print(clinical_md['Obesity'].sum())

pec
55
ghtn
49
gdm
114
stillbirth
15
chtn
71

past pec
67
ivf
17
age
29.90762991097281 6.037320565527862
first preg
484
obesity
447


### NYU

In [ ]:
def proc_on_no(df: pd.DataFrame, col: str) -> pd.DataFrame:
    def parse(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, str):
            if 'no' in x.lower():
                return(0)
            else:
                return(1)
        
    df[col] = df[col].apply(parse)
    return df

def get_bmi(df: pd.DataFrame, weight_col: str, height_col: str) -> pd.DataFrame:
    df['bmi'] = df[weight_col]/(df[height_col]**2)
    return(df)

In [ ]:
nyu_clinical_md = pd.read_csv('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/nyu_ml_clinical_features_20260501.csv', index_col=0)
nyu_pec = pd.read_pickle("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/nyu_cases_051126.pkl")
nyu_controls = pd.read_pickle("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/nyu_controls_051126.pkl")
nyu_ids = nyu_pec+nyu_controls

nyu_clinical_md = nyu_clinical_md[nyu_clinical_md['id'].isin(nyu_ids)]

nyu_ghtn_ids = ['nyu017', 'nyu019', 'nyu032', 'nyu039', 'nyu041', 'nyu069', 'nyu082', 'nyu054']
nyu_gdm_ids = ['nyu007', 'nyu015', 'nyu079', 'nyu080']

participant_tracker = pd.read_csv('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/Visionary AI Participant Tracker(FMF Algorithim )_052126.csv')
participant_tracker = participant_tracker[participant_tracker['Record ID']!='NYU130'] # stillbirth
participant_tracker = participant_tracker.drop(columns=[x for x in participant_tracker.columns if 'Unnamed' in x])

nyu_md = participant_tracker.copy()
nyu_md =nyu_md.drop_duplicates()

nyu_md['Record ID'] = nyu_md['Record ID'].str.lower()
nyu_md = nyu_md[nyu_md['Record ID'].isin(nyu_ids)]



nyu_md = get_bmi(nyu_md, 'Weight (kg)', 'Height (m)')

nyu_md['obesity'] = nyu_md['bmi']>=30


nyu_md = proc_on_no(nyu_md, 'Nulliparity?')

In [ ]:
def compare_binary_col(
    nyu_md,
    col,
    controls_ids,
    group_ids,
    id_col="Record ID"
):
    """
    Parameters
    ----------
    nyu_md : DataFrame
    col : str
        Binary column to compare.
    controls_ids : list
        IDs for controls.
    group_ids : dict
        Dict of group_name -> list_of_ids.
    """
    ids = nyu_md[id_col].str.lower()
    controls = nyu_md[ids.isin(controls_ids)]
    control_yes = int(controls[col].sum())
    control_no = len(controls) - control_yes
    print(f"\n=== {col} ===")
    print(f"controls: {control_yes}/{len(controls)} "
          f"({control_yes/len(controls):.3f})")
    results = []
    for group_name, group_list in group_ids.items():
        df = nyu_md[ids.isin(group_list)]
        yes = int(df[col].sum())
        no = len(df) - yes
        prop = yes / len(df)

        odds_ratio, pval = fisher_exact([
            [control_yes, control_no],
            [yes, no]
        ])

        print(f"{group_name}: {yes}/{len(df)} ({prop:.3f}) pval: {pval}")
        results.append({
            "group": group_name,
            "n": len(df),
            "count": yes,
            "proportion": prop,
            "odds_ratio": odds_ratio,
            "pvalue": pval
        })
    return pd.DataFrame(results)



In [11]:

print(len(nyu_pec))
print(len([x for x in nyu_ghtn_ids if x in nyu_ids]))
print(len([x for x in nyu_gdm_ids if x in nyu_ids]))
print('no stillbirths')
print()
print(nyu_clinical_md['CHTN'].sum())
print(nyu_clinical_md['Past PEC'].sum())
print(nyu_clinical_md['Maternal Diag IVF'].sum())
print(nyu_clinical_md['Age at enrollment'].mean(),nyu_clinical_md['Age at enrollment'].std())


14
8
4
no stillbirths

5.0
5
18
34.73134328358209 4.53465474130006


In [9]:
compare_binary_col(
    nyu_md=nyu_md,
    col='Nulliparity?',
    controls_ids=nyu_controls,
    group_ids={'cohort': nyu_ids,'pec':nyu_pec},
    id_col="Record ID"
)


=== Nulliparity? ===
controls: 40/65 (0.615)
cohort: 51/79 (0.646) pval: 0.7312368437716523
pec: 11/14 (0.786) pval: 0.3566148889911351


,group,n,count,proportion,odds_ratio,pvalue
0,cohort,79,51,0.645570,0.878431,0.731237
1,pec,14,11,0.785714,0.436364,0.356615


In [10]:
compare_binary_col(
    nyu_md=nyu_md,
    col='obesity',
    controls_ids=nyu_controls,
    group_ids={'cohort': nyu_ids,'pec':nyu_pec},
    id_col="Record ID"
)


=== obesity ===
controls: 16/65 (0.246)
cohort: 21/79 (0.266) pval: 0.849360774365261
pec: 5/14 (0.357) pval: 0.5056162260655342


,group,n,count,proportion,odds_ratio,pvalue
0,cohort,79,21,0.265823,0.901846,0.849361
1,pec,14,5,0.357143,0.587755,0.505616
